# QStorePrice AI — Complete End-to-End Kaggle Pipeline

**Project**: RL-trained LLM (Qwen-2.5) that writes *Operating Briefs* to manage perishable goods in quick-commerce stores. Unified metric: **WRR (Weekly Waste Recovery Rate)**.

**Pipeline**: GitHub clone → Install → Env verify → SFT → GRPO rollouts → DPO → Eval → Backend server → Push to HF Hub

**Hardware target**: Kaggle T4 (16 GB VRAM) or P100 (16 GB VRAM)

**HF credits**: Used for (1) HF Inference API during GRPO rollouts (optional, faster) and (2) pushing the trained model to HF Hub.

---
### ⚙️ Configuration — Change these before running
| Variable | Default | Notes |
|---|---|---|
| `HF_TOKEN` | required | Your HF token with write access |
| `MODEL_ID` | `Qwen/Qwen2.5-1.5B-Instruct` | Use `7B` for better quality (slower) |
| `USE_HF_INFERENCE_API` | `False` | `True` = use HF credits for GRPO inference |
| `GRPO_EPISODES` | `5` | Increase for better training (slow) |
| `HF_REPO_ID` | set below | Where to push your trained model |

In [ ]:
# ============================================================
# CELL 1 — USER CONFIGURATION
# Edit these values before running the notebook.
# ============================================================

import os

# --- Hugging Face ---
HF_TOKEN      = "hf_REPLACE_WITH_YOUR_TOKEN"   # HF token with read+write access
HF_REPO_ID    = "your-hf-username/qstoreprice-sft"  # where to push the model

# --- Model ---
# Qwen2.5-1.5B: fast, fits T4, good for demo (~20 min SFT)
# Qwen2.5-7B:   better quality, fits T4 in 4-bit, longer (~60 min SFT)
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# --- Training ---
SFT_EPOCHS       = 2        # 2 epochs is enough for format learning
GRPO_EPISODES    = 5        # episodes for rollout collection (increase to 20+ for real training)
DPO_MIN_PAIRS    = 4        # min DPO pairs needed before DPO runs
SEED             = 42

# --- HF Inference API ---
# True  = uses your HF credits for GRPO inference (faster, uses ~0.05 credits/episode)
# False = uses local GPU for inference (free, slower)
USE_HF_INFERENCE_API = False

# --- Paths (Kaggle standard) ---
WORK_DIR         = "/kaggle/working"
REPO_DIR         = f"{WORK_DIR}/QStorePrice"
CHECKPOINTS_DIR  = f"{WORK_DIR}/checkpoints"
SFT_DIR          = f"{CHECKPOINTS_DIR}/sft_v1"
GRPO_DIR         = f"{CHECKPOINTS_DIR}/grpo_level0"
DPO_DIR          = f"{CHECKPOINTS_DIR}/dpo_round1"
FINAL_DIR        = f"{CHECKPOINTS_DIR}/final"
PLOTS_DIR        = f"{WORK_DIR}/plots"

os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

print("Configuration loaded.")
print(f"  MODEL_ID            : {MODEL_ID}")
print(f"  GRPO_EPISODES       : {GRPO_EPISODES}")
print(f"  USE_HF_INFERENCE_API: {USE_HF_INFERENCE_API}")
print(f"  REPO_DIR            : {REPO_DIR}")
print(f"  CHECKPOINTS_DIR     : {CHECKPOINTS_DIR}")

## Section 1 — Hardware & Environment Check

In [ ]:
# ============================================================
# CELL 2 — GPU / HARDWARE CHECK
# ============================================================

import subprocess, sys, platform

print("=" * 55)
print(" QStorePrice AI — Kaggle Hardware Check")
print("=" * 55)

# Python version
print(f"Python : {sys.version.split()[0]}")
print(f"OS     : {platform.system()} {platform.release()}")

# GPU check
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"PyTorch: {torch.__version__}")
    if cuda_ok:
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU    : {gpu_name}")
        print(f"VRAM   : {vram_gb:.1f} GB")
        if vram_gb < 14:
            print("WARNING: < 14 GB VRAM — training may OOM. Try 1.5B model.")
    else:
        print("GPU    : NOT AVAILABLE — enable GPU accelerator in Kaggle settings")
except ImportError:
    print("PyTorch not yet installed (will be installed below).")

# Disk space
result = subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True)
print("\nDisk usage (working dir):")
print(result.stdout.strip())

# nvidia-smi summary
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                         "--format=csv,noheader"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"\nnvidia-smi: {result.stdout.strip()}")

## Section 2 — Install Dependencies

In [ ]:
# ============================================================
# CELL 3 — INSTALL UNSLOTH (must be first, Kaggle-specific)
# ============================================================
# Unsloth has a special Kaggle install. This must happen before
# other ML packages to avoid version conflicts.

import subprocess, sys

def run(cmd, desc=""):
    """Run a shell command and print output."""
    print(f"\n>>> {desc or cmd[:80]}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        last_lines = result.stdout.strip().split("\n")[-5:]
        print("\n".join(last_lines))
    if result.returncode != 0:
        print(f"STDERR: {result.stderr.strip()[-300:]}")
        raise RuntimeError(f"Command failed: {cmd}")
    return result

# Step 1: Install Unsloth with Kaggle-optimized build
run(
    'pip install -q "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"',
    "Installing Unsloth (Kaggle build)"
)

print("\nUnsloth installed successfully.")

In [ ]:
# ============================================================
# CELL 4 — INSTALL REMAINING DEPENDENCIES
# ============================================================

deps = [
    # Core RL + training stack
    "transformers>=4.40.0",
    "trl>=0.8.6",
    "peft>=0.10.0",
    "accelerate>=0.29.3",
    "bitsandbytes>=0.43.1",
    "datasets>=2.19.0",
    # Environment
    "gymnasium>=0.29.1,<1.0",
    "pydantic>=2.0.0",
    # Experiment tracking
    "wandb>=0.17.0",
    # HF Hub utilities
    "huggingface_hub>=0.22.0",
    # Utilities
    "numpy>=1.26.0",
    "pandas>=2.2.1",
    "scipy>=1.13.0",
    "tqdm>=4.66.2",
    "python-dotenv>=1.0.1",
    "matplotlib>=3.7.0",
    # Backend server
    "fastapi>=0.110.0",
    "uvicorn>=0.27.0",
    "httpx>=0.27.0",  # async HTTP client for server testing
]

import subprocess

pkg_str = " ".join(f'"{d}"' for d in deps)
result = subprocess.run(
    f"pip install -q {pkg_str}",
    shell=True, capture_output=True, text=True
)
last = result.stdout.strip().split("\n")[-3:]
print("\n".join(last))
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])
    raise RuntimeError("Dependency installation failed")

# Try openenv-core (OpenEnv hackathon package)
oe = subprocess.run("pip install -q openenv-core>=0.2.0", shell=True,
                    capture_output=True, text=True)
if oe.returncode == 0:
    print("openenv-core installed.")
else:
    print("openenv-core not available on PyPI — server section will use fallback mode.")

print("\nAll dependencies installed.")

## Section 3 — Clone Repository & Setup

In [ ]:
# ============================================================
# CELL 5 — CLONE REPO AND SET UP PYTHON PATH
# ============================================================

import os, sys, subprocess

WORK_DIR  = "/kaggle/working"
REPO_DIR  = f"{WORK_DIR}/QStorePrice"

# Clone (or pull if already present — handles re-runs)
if os.path.exists(REPO_DIR):
    print(f"Repo already cloned at {REPO_DIR}. Pulling latest...")
    r = subprocess.run("git pull", cwd=REPO_DIR, capture_output=True, text=True, shell=True)
    print(r.stdout.strip() or "Already up to date.")
else:
    print("Cloning QStorePrice repository...")
    r = subprocess.run(
        "git clone https://github.com/nandeshkanagaraju/QStorePrice.git",
        cwd=WORK_DIR, capture_output=True, text=True, shell=True
    )
    if r.returncode != 0:
        raise RuntimeError(f"Clone failed: {r.stderr}")
    print("Clone complete.")

# Add repo root to Python path so all internal imports resolve
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Verify key directories exist
for subdir in ["freshprice_env", "training", "eval", "server", "training/sft_data"]:
    path = os.path.join(REPO_DIR, subdir)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  {subdir:30s} [{status}]")

# Print commit hash for reproducibility
r = subprocess.run("git log --oneline -3", cwd=REPO_DIR,
                   capture_output=True, text=True, shell=True)
print(f"\nRecent commits:\n{r.stdout.strip()}")
print(f"\nRepo root in sys.path: {REPO_DIR}")

## Section 4 — Hugging Face Authentication

In [ ]:
# ============================================================
# CELL 6 — HF AUTHENTICATION
# ============================================================
# Option A: Token is set manually above in CELL 1.
# Option B: Use Kaggle Secrets (recommended — avoids token in notebook).
#   1. Kaggle sidebar → Add-ons → Secrets → Add "HF_TOKEN"
#   2. Uncomment the two lines below.

# from kaggle_secrets import UserSecretsClient
# HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

import os
from huggingface_hub import login, whoami

if HF_TOKEN == "hf_REPLACE_WITH_YOUR_TOKEN":
    raise ValueError(
        "Set HF_TOKEN in Cell 1 (or use Kaggle Secrets as shown above).\n"
        "Get your token at: https://huggingface.co/settings/tokens"
    )

# Log in to HF Hub
login(token=HF_TOKEN, add_to_git_credential=False)

# Verify who we are
user_info = whoami(token=HF_TOKEN)
print(f"Logged in as: {user_info['name']}")
print(f"Account type: {user_info.get('type', 'user')}")

# Set env var so transformers / datasets auto-authenticate
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

# Disable WandB telemetry (optional, avoids login prompt)
os.environ["WANDB_DISABLED"] = "true"

print("\nHF authentication complete.")

## Section 5 — Environment Smoke Test

In [ ]:
# ============================================================
# CELL 7 — FRESHPRICE ENVIRONMENT SMOKE TEST
# Verifies all modules import and the Gym env boots correctly.
# Expected: reset() returns a 1000-5000 char observation string.
# ============================================================

import sys, os
sys.path.insert(0, REPO_DIR)  # safety: ensure path is set

print("Importing FreshPriceEnv...")
from freshprice_env.freshprice_env import FreshPriceEnv
from freshprice_env.enums import CurriculumScenario
from freshprice_env.constants import TOTAL_TICKS, BRIEFS_PER_EPISODE, TICKS_PER_BRIEF

print("\n--- Episode Constants ---")
print(f"  Total ticks/episode : {TOTAL_TICKS}")
print(f"  Briefs/episode      : {BRIEFS_PER_EPISODE}")
print(f"  Ticks per brief     : {TICKS_PER_BRIEF}")
print(f"  Curriculum scenarios: {[s.name for s in CurriculumScenario]}")

# Test each scenario initialises
print("\n--- Scenario Reset Tests ---")
for scenario in CurriculumScenario:
    env = FreshPriceEnv(scenario=scenario, seed=42)
    obs, info = env.reset()
    state = env.state()
    print(
        f"  {scenario.name:20s} | obs_len={len(obs):5d} | "
        f"batches={state['active_batches']} | "
        f"engine={info['engine_type']}"
    )
    assert len(obs) > 100, f"Observation too short for {scenario.name}"

# Run a few manual steps on STABLE_WEEK with a hardcoded brief
print("\n--- Manual 3-Step Walkthrough (STABLE_WEEK) ---")
env = FreshPriceEnv(scenario=CurriculumScenario.STABLE_WEEK, seed=42)
obs, info = env.reset()
print(f"Initial engine: {info['engine_type']}")
print(f"Observation excerpt (first 400 chars):\n{obs[:400]}")

# A minimal but valid Operating Brief
FALLBACK_BRIEF = """\
## SITUATION SUMMARY
Reviewing current inventory for pricing optimization.

## SIGNAL ANALYSIS
Some batches approaching expiry. Moderate demand velocity.

## VIABILITY CHECK
Small discounts on near-expiry stock should recover cost.

## RECOMMENDATION
Apply conservative discounts to URGENT batches.

## DIRECTIVE
{"engine": "PRICING", "actions": []}

## CONFIDENCE
0.6
"""

total_reward = 0.0
for step in range(3):
    obs, reward, done, truncated, info = env.step(FALLBACK_BRIEF)
    total_reward += reward
    print(
        f"  Step {step+1}: reward={reward:+.4f} | "
        f"parse={'OK' if info['parse_success'] else 'FAIL'} | "
        f"quality={info['quality_score']:.3f} | "
        f"next_engine={info.get('next_engine_type','END')}"
    )

print(f"\n3-step cumulative reward: {total_reward:+.4f}")
print("\nEnvironment smoke test PASSED.")

## Section 6 — SFT Data Generation & Inspection


In [ ]:
# ============================================================
# CELL 8a — GENERATE SFT TRAINING DATA
# Creates training/sft_data/ with 90 examples (10 per difficulty
# level × 3 engines). Safe to re-run — overwrites existing files.
# ============================================================

import sys, os
sys.path.insert(0, REPO_DIR)

from training.generate_sft_data import generate_all

sft_data_dir = os.path.join(REPO_DIR, "training", "sft_data")
print("Generating SFT training data...")
generate_all(output_dir=sft_data_dir, n_per_difficulty=10)


In [ ]:
# ============================================================
# CELL 8b — INSPECT SFT TRAINING DATA
# Verifies all 90 examples were written and have the 6 required
# section headers. Runs after CELL 8a.
# ============================================================

import json
from pathlib import Path

sft_data_dir = Path(REPO_DIR) / "training" / "sft_data"
print("SFT data files:")

all_examples = []
if not sft_data_dir.exists():
    print(f"  [!] Directory not found: {sft_data_dir}")
    print("  Run CELL 8a first to generate the data.")
else:
    for json_file in sorted(sft_data_dir.glob("*.json")):
        with open(json_file) as f:
            examples = json.load(f)
        all_examples.extend(examples)
        engine_counts = {}
        for ex in examples:
            et = ex.get("engine_type", "UNKNOWN")
            engine_counts[et] = engine_counts.get(et, 0) + 1
        print(f"  {json_file.name:30s} {len(examples):3d} examples | {engine_counts}")

print(f"\nTotal SFT examples: {len(all_examples)}")

difficulty = {}
for ex in all_examples:
    d = ex.get("difficulty", "unknown")
    difficulty[d] = difficulty.get(d, 0) + 1
print(f"Difficulty split: {difficulty}")

if not all_examples:
    print("\n[!] No examples loaded — skipping display and validation.")
else:
    print("\n--- Example #1 (truncated) ---")
    ex = all_examples[0]
    print(f"Engine type : {ex.get('engine_type')}")
    print(f"Difficulty  : {ex.get('difficulty')}")
    print(f"Prompt len  : {len(ex.get('prompt', ''))} chars")
    print(f"Completion  : {ex.get('completion', '')[:400]}...")

    REQUIRED = ["SITUATION:", "SIGNAL ANALYSIS:", "VIABILITY CHECK:",
                "RECOMMENDATION:", "DIRECTIVE:", "CONFIDENCE:"]
    bad = []
    for i, ex in enumerate(all_examples):
        comp = ex.get("completion", "")
        missing = [s for s in REQUIRED if s not in comp]
        if missing:
            bad.append((i, missing))

    if bad:
        print(f"\nWARNING: {len(bad)} examples missing required sections:")
        for idx, miss in bad[:5]:
            print(f"  Example {idx}: missing {miss}")
    else:
        print(f"\nAll {len(all_examples)} examples contain all 6 required sections.")


## Section 7 — SFT Warm-Start Training

In [ ]:
# ============================================================
# CELL 9 — RUN SFT WARM-START
# Teaches the model the 6-section Operating Brief format.
# Expected duration:
#   1.5B model, 2 epochs, T4: ~20 min
#   7B model, 2 epochs, T4:   ~60 min
# ============================================================

import os, sys, torch
sys.path.insert(0, REPO_DIR)

from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

from training.sft_trainer import load_sft_dataset
from freshprice_env.brief_pipeline.prompt_builder import OperatingBriefPromptBuilder

# ---------- Load model ----------
print(f"Loading {MODEL_ID} in 4-bit with Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
    token=HF_TOKEN,
)
print(f"Model loaded. Trainable params before LoRA:")
total = sum(p.numel() for p in model.parameters())
print(f"  Total params: {total/1e6:.1f}M")

# ---------- Add LoRA adapters ----------
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  LoRA trainable params: {trainable/1e6:.2f}M ({trainable/total*100:.1f}% of total)")

# ---------- Load + format dataset ----------
dataset = load_sft_dataset(data_dir=os.path.join(REPO_DIR, "training", "sft_data"))
print(f"\nDataset: {len(dataset)} examples")

# ---------- Configure SFTTrainer ----------
os.makedirs(SFT_DIR, exist_ok=True)

training_args = SFTConfig(
    output_dir=SFT_DIR,
    num_train_epochs=SFT_EPOCHS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    save_steps=100,
    seed=SEED,
    max_seq_length=4096,
    dataset_text_field="text",
    report_to="none",  # WandB disabled in Kaggle
    packing=True,      # Unsloth packing for efficiency
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

# ---------- Train ----------
print("\nStarting SFT training...")
trainer_stats = trainer.train()

print("\n=== SFT Training Complete ===")
print(f"  Training loss    : {trainer_stats.training_loss:.4f}")
print(f"  Runtime          : {trainer_stats.metrics.get('train_runtime', 0):.1f}s")
print(f"  Samples/sec      : {trainer_stats.metrics.get('train_samples_per_second', 0):.2f}")
print(f"  Epochs completed : {SFT_EPOCHS}")

# ---------- Save merged 16-bit checkpoint ----------
print(f"\nSaving merged 16-bit checkpoint to {SFT_DIR}...")
model.save_pretrained_merged(
    SFT_DIR,
    tokenizer,
    save_method="merged_16bit",
)
print(f"Checkpoint saved: {SFT_DIR}")

# ---------- Quick generation test ----------
print("\nRunning generation sanity check...")
FastLanguageModel.for_inference(model)

test_prompt = (
    f"<|im_start|>system\n{OperatingBriefPromptBuilder.SYSTEM_PROMPT}<|im_end|>\n"
    "<|im_start|>user\n"
    "=== CURRENT INVENTORY ===\n"
    "[🔴] dairy — Batch B001\n"
    "  Stock: 8 units | Expiry: 4.0hrs | Urgency: CRITICAL\n"
    "  Current price: Rs 80 | Original: Rs 80 | Floor: Rs 35\n"
    "=== YOUR TASK ===\nWrite a PRICING Operating Brief.\n"
    "<|im_end|>\n<|im_start|>assistant\n"
)

inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True,
                   max_length=2048).to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs, max_new_tokens=500, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:],
                              skip_special_tokens=True)

REQUIRED_SECTIONS = ["SITUATION:", "SIGNAL ANALYSIS:", "VIABILITY CHECK:",
                     "RECOMMENDATION:", "DIRECTIVE:", "CONFIDENCE:"]
missing = [s for s in REQUIRED_SECTIONS if s not in generated]

print("\nGenerated brief excerpt:")
print(generated[:600])
print()

if missing:
    print(f"WARNING: Missing sections in generated brief: {missing}")
    print("Model may need more SFT epochs. Consider increasing SFT_EPOCHS to 3.")
else:
    print(f"All {len(REQUIRED_SECTIONS)} sections present. SFT checkpoint VERIFIED.")

# Store reference to SFT checkpoint
CURRENT_CHECKPOINT = SFT_DIR
print(f"\nCurrent checkpoint: {CURRENT_CHECKPOINT}")

## Section 8 — GRPO Episode Rollouts

In [ ]:
# ============================================================
# CELL 10 — DEFINE INFERENCE BACKEND
# Chooses between HF Inference API (uses HF credits) or local GPU.
# ============================================================

import sys
sys.path.insert(0, REPO_DIR)

class LocalInferenceClient:
    """
    Local GPU inference using the SFT model already loaded in memory.
    Called by FreshPriceEnv via llm_client=self.
    """
    def __init__(self, model, tokenizer, system_prompt):
        from unsloth import FastLanguageModel
        FastLanguageModel.for_inference(model)
        self._model = model
        self._tok   = tokenizer
        self._sys   = system_prompt

    def generate(self, prompt: str) -> str:
        import torch
        full = (
            f"<|im_start|>system\n{self._sys}<|im_end|>\n"
            f"<|im_start|>user\n{prompt}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        inputs = self._tok(full, return_tensors="pt", truncation=True,
                           max_length=3800).to(self._model.device)
        with torch.no_grad():
            out = self._model.generate(
                **inputs, max_new_tokens=600,
                temperature=0.7, do_sample=True,
                pad_token_id=self._tok.eos_token_id,
            )
        return self._tok.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )


class HFAPIInferenceClient:
    """
    HF Inference API client — uses your HF credits.
    Each call = ~1-3 API requests.
    At ~0.05 credits/episode, 30 credits ≈ 600 episodes max.
    """
    def __init__(self, model_id, token, system_prompt):
        from huggingface_hub import InferenceClient
        self._client = InferenceClient(model=model_id, token=token)
        self._model  = model_id
        self._sys    = system_prompt

    def generate(self, prompt: str) -> str:
        messages = [
            {"role": "system", "content": self._sys},
            {"role": "user",   "content": prompt},
        ]
        response = self._client.chat_completion(
            messages=messages,
            max_tokens=600,
            temperature=0.7,
        )
        return response.choices[0].message.content


from freshprice_env.brief_pipeline.prompt_builder import OperatingBriefPromptBuilder

# Build the inference client based on config
if USE_HF_INFERENCE_API:
    print(f"Using HF Inference API: {MODEL_ID}")
    print("NOTE: Each GRPO episode uses ~0.05 HF credits.")
    llm_client = HFAPIInferenceClient(
        model_id=MODEL_ID,
        token=HF_TOKEN,
        system_prompt=OperatingBriefPromptBuilder.SYSTEM_PROMPT,
    )
else:
    print("Using local GPU inference (free, slower).")
    # model was loaded in Cell 9; re-use it
    llm_client = LocalInferenceClient(
        model=model,
        tokenizer=tokenizer,
        system_prompt=OperatingBriefPromptBuilder.SYSTEM_PROMPT,
    )

# Quick test call
test_out = llm_client.generate(
    "=== QUICK TEST ===\nWrite one sentence about pricing perishables."
)
print(f"\nTest generation: {test_out[:200]}")
print("\nInference client ready.")

In [ ]:
# ============================================================
# CELL 11 — GRPO EPISODE ROLLOUTS
# Runs GRPO_EPISODES episodes, collects trajectories for DPO.
# The environment is deterministic given seed — fully reproducible.
#
# NOTE: This is rollout collection (inference only).
# Gradient updates happen in the DPO step below.
#
# Expected time per episode:
#   1.5B local: ~8-12 min | 7B local: ~25-35 min
#   HF API:     ~2-4 min (latency dependent)
# ============================================================

import sys, os, json, time, random
sys.path.insert(0, REPO_DIR)

from freshprice_env.freshprice_env import FreshPriceEnv
from freshprice_env.enums import CurriculumScenario
from training.curriculum import CurriculumManager, EpisodeResult
from training.trajectory_buffer import Trajectory, TrajectoryBuffer
from training.counterfactual import CounterfactualEngine

# Initialise managers
rng                 = random.Random(SEED)
curriculum          = CurriculumManager()
trajectory_buffer   = TrajectoryBuffer(rng=rng)
counterfactual_eng  = CounterfactualEngine(rng=rng)

episode_results = []  # for plotting later
scenario = CurriculumScenario.STABLE_WEEK  # Level 0

print(f"Starting GRPO rollouts: {GRPO_EPISODES} episodes on {scenario.name}")
print(f"{'─'*60}")
print(f"{'Ep':>4} {'WRR':>6} {'R1-P':>6} {'R2-F':>6} {'R3-T':>6} {'Qual':>6} "
      f"{'Viol':>5} {'Const':>6} {'Time':>6}")
print(f"{'─'*60}")

total_start = time.time()

for ep_idx in range(GRPO_EPISODES):
    ep_seed = rng.randint(0, 999999)
    ep_start = time.time()

    # Initialise fresh environment for this episode
    env = FreshPriceEnv(
        scenario=scenario,
        seed=ep_seed,
        llm_client=llm_client,
    )
    obs, info = env.reset(seed=ep_seed)

    ep_briefs   = []
    done        = False
    step_count  = 0

    while not done:
        # Generate brief using the LLM client
        raw_brief = llm_client.generate(obs)
        obs, reward, done, truncated, info = env.step(raw_brief)

        ep_briefs.append({
            "tick":         info.get("tick", step_count * 8),
            "engine_type":  info.get("engine_type", "PRICING"),
            "raw_response": raw_brief,
            "quality_score":info.get("quality_score", 0.0),
            "reward_delta": reward,
            "parse_success":info.get("parse_success", True),
        })
        step_count += 1

    # Extract final episode metrics
    final_reward = info.get("final_reward", {})
    audit        = info.get("constitutional_audit", {})

    wrr            = final_reward.get("wrr", 0.0)
    r1             = final_reward.get("r1_pricing", 0.0)
    r2             = final_reward.get("r2_farmer", 0.0)
    r3             = final_reward.get("r3_trend", 0.0)
    quality        = final_reward.get("brief_quality_score", 0.0)
    violations     = final_reward.get("anti_hack_violations", 0)
    valid          = final_reward.get("episode_valid", True)
    const_passed   = audit.get("passed", True)
    elapsed        = time.time() - ep_start

    ep_result = {
        "episode": ep_idx, "seed": ep_seed, "wrr": wrr,
        "r1": r1, "r2": r2, "r3": r3,
        "quality": quality, "violations": violations,
        "valid": valid, "constitutional_passed": const_passed,
        "steps": step_count, "elapsed_sec": elapsed,
    }
    episode_results.append(ep_result)

    # Add to trajectory buffer if clean
    if valid and const_passed:
        traj = Trajectory(
            episode_num=ep_idx,
            scenario=scenario,
            wrr=wrr,
            brief_quality_score=quality,
            constitutional_passed=const_passed,
            episode_valid=valid,
            briefs=ep_briefs,
            reward_engine_snapshot=final_reward,
        )
        trajectory_buffer.add(traj)

    # Update curriculum
    curriculum.record_episode(EpisodeResult(
        episode_num=ep_idx, scenario=scenario, wrr=wrr,
        brief_quality_score=quality, anti_hack_violations=violations,
        constitutional_passed=const_passed, episode_valid=valid,
    ))

    print(
        f"{ep_idx+1:>4} {wrr:>6.3f} {r1:>6.3f} {r2:>6.3f} {r3:>6.3f} "
        f"{quality:>6.3f} {violations:>5} {'PASS' if const_passed else 'FAIL':>6} "
        f"{elapsed:>5.0f}s"
    )

print(f"{'─'*60}")
total_elapsed = time.time() - total_start

# Summary
wrrs    = [e["wrr"] for e in episode_results]
avg_wrr = sum(wrrs) / len(wrrs)
max_wrr = max(wrrs)
buf_size = trajectory_buffer.get_stats()["buffer_size"]

print(f"\nGRPO Rollouts Summary")
print(f"  Episodes completed  : {GRPO_EPISODES}")
print(f"  Avg WRR             : {avg_wrr:.4f}")
print(f"  Max WRR             : {max_wrr:.4f}")
print(f"  Buffer size         : {buf_size}")
print(f"  Total time          : {total_elapsed/60:.1f} min")

if avg_wrr >= 0.70:
    print(f"  STATUS: PROMOTION ELIGIBLE (WRR >= 0.70)")
else:
    print(f"  STATUS: Need {0.70 - avg_wrr:.3f} more WRR to reach promotion threshold.")

# Save episode log
log_path = os.path.join(WORK_DIR, "episode_log.json")
with open(log_path, "w") as f:
    json.dump(episode_results, f, indent=2)
print(f"\nEpisode log saved: {log_path}")

## Section 9 — DPO Fine-tuning

In [ ]:
# ============================================================
# CELL 12 — DPO FINE-TUNING
# Uses the trajectory buffer to generate preference pairs.
# Requires >= DPO_MIN_PAIRS clean episodes in the buffer.
# ============================================================

import sys, os, torch
sys.path.insert(0, REPO_DIR)

from training.dpo_trainer import run_dpo

buf_stats  = trajectory_buffer.get_stats()
buf_size   = buf_stats["buffer_size"]

print(f"Trajectory buffer: {buf_size} clean episodes")

if buf_size < DPO_MIN_PAIRS:
    print(
        f"Skipping DPO — buffer has {buf_size} episodes, "
        f"need >= {DPO_MIN_PAIRS}.\n"
        f"To get more pairs: increase GRPO_EPISODES or lower DPO_MIN_PAIRS."
    )
    CURRENT_CHECKPOINT = SFT_DIR  # use SFT checkpoint
else:
    print(f"Generating DPO preference pairs...")
    dpo_pairs = trajectory_buffer.generate_dpo_pairs(counterfactual_eng)
    print(f"Generated {len(dpo_pairs)} DPO pairs.")

    if len(dpo_pairs) < DPO_MIN_PAIRS:
        print(f"Not enough pairs ({len(dpo_pairs)}). Skipping DPO.")
        CURRENT_CHECKPOINT = SFT_DIR
    else:
        print(f"\nRunning DPO fine-tuning on {len(dpo_pairs)} pairs...")
        print(f"  Input checkpoint : {SFT_DIR}")
        print(f"  Output checkpoint: {DPO_DIR}")

        os.makedirs(DPO_DIR, exist_ok=True)

        new_checkpoint = run_dpo(
            checkpoint_dir=SFT_DIR,
            output_dir=DPO_DIR,
            dpo_pairs=dpo_pairs,
            seed=SEED,
        )

        CURRENT_CHECKPOINT = new_checkpoint
        print(f"\nDPO complete. Checkpoint: {CURRENT_CHECKPOINT}")

## Section 10 — Evaluation

In [ ]:
# ============================================================
# CELL 13 — DETERMINISTIC EVALUATION
# Runs 3 episodes per scenario with greedy decoding.
# Fixed seeds ensure reproducible results.
# ============================================================

import sys, os, json, time
sys.path.insert(0, REPO_DIR)

from freshprice_env.freshprice_env import FreshPriceEnv
from freshprice_env.enums import CurriculumScenario

# For evaluation: use greedy decoding (deterministic)
class GreedyClient:
    """Greedy-decoding wrapper around the local model."""
    def __init__(self, model, tokenizer, system_prompt):
        from unsloth import FastLanguageModel
        FastLanguageModel.for_inference(model)
        self._model = model
        self._tok   = tokenizer
        self._sys   = system_prompt

    def generate(self, prompt: str) -> str:
        import torch
        full = (
            f"<|im_start|>system\n{self._sys}<|im_end|>\n"
            f"<|im_start|>user\n{prompt}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        inputs = self._tok(full, return_tensors="pt", truncation=True,
                           max_length=3800).to(self._model.device)
        with torch.no_grad():
            out = self._model.generate(
                **inputs, max_new_tokens=600,
                do_sample=False,  # greedy
                pad_token_id=self._tok.eos_token_id,
            )
        return self._tok.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )

from freshprice_env.brief_pipeline.prompt_builder import OperatingBriefPromptBuilder
greedy_client = GreedyClient(
    model=model, tokenizer=tokenizer,
    system_prompt=OperatingBriefPromptBuilder.SYSTEM_PROMPT,
)

# Evaluate on 2 scenarios (STABLE + CRISIS) with 3 seeds each
eval_scenarios = [
    CurriculumScenario.STABLE_WEEK,
    CurriculumScenario.CRISIS_WEEK,
]
EVAL_SEEDS_PER_SCENARIO = 3

eval_results = {}
print("=" * 60)
print(" EVALUATION REPORT")
print(f" Checkpoint: {CURRENT_CHECKPOINT}")
print("=" * 60)

for scenario in eval_scenarios:
    level  = scenario.value
    seeds  = [level * 1000 + i for i in range(EVAL_SEEDS_PER_SCENARIO)]
    ep_res = []

    print(f"\n-- {scenario.name} ({EVAL_SEEDS_PER_SCENARIO} episodes) --")

    for seed in seeds:
        env  = FreshPriceEnv(scenario=scenario, seed=seed, llm_client=greedy_client)
        obs, info = env.reset(seed=seed)
        done = False

        while not done:
            brief = greedy_client.generate(obs)
            obs, reward, done, truncated, info = env.step(brief)

        fr    = info.get("final_reward", {})
        audit = info.get("constitutional_audit", {})

        ep_res.append({
            "seed": seed,
            "wrr":     fr.get("wrr", 0.0),
            "r1":      fr.get("r1_pricing", 0.0),
            "r2":      fr.get("r2_farmer", 0.0),
            "r3":      fr.get("r3_trend", 0.0),
            "quality": fr.get("brief_quality_score", 0.0),
            "violations": fr.get("anti_hack_violations", 0),
            "constitutional_passed": audit.get("passed", True),
        })

        print(
            f"  seed={seed}: WRR={ep_res[-1]['wrr']:.4f} | "
            f"quality={ep_res[-1]['quality']:.4f} | "
            f"violations={ep_res[-1]['violations']} | "
            f"const={'PASS' if ep_res[-1]['constitutional_passed'] else 'FAIL'}"
        )

    wrrs = [e["wrr"] for e in ep_res]
    mean_wrr = sum(wrrs) / len(wrrs)
    import math
    std_wrr  = math.sqrt(sum((w - mean_wrr)**2 for w in wrrs) / max(len(wrrs)-1,1))

    eval_results[scenario.name] = {
        "wrr_mean": round(mean_wrr, 4),
        "wrr_std":  round(std_wrr, 4),
        "wrr_min":  round(min(wrrs), 4),
        "wrr_max":  round(max(wrrs), 4),
        "quality":  round(sum(e["quality"] for e in ep_res) / len(ep_res), 4),
        "violations_mean": round(sum(e["violations"] for e in ep_res) / len(ep_res), 1),
        "constitutional_pass_rate": f"{sum(e['constitutional_passed'] for e in ep_res)}/{len(ep_res)}",
    }

    r = eval_results[scenario.name]
    print(f"\n  WRR  : {r['wrr_mean']:.4f} ± {r['wrr_std']:.4f}  [{r['wrr_min']:.4f} → {r['wrr_max']:.4f}]")
    print(f"  Qual : {r['quality']:.4f}")
    print(f"  Viol : {r['violations_mean']} avg/episode")
    print(f"  Const: {r['constitutional_pass_rate']}")

# Save eval results
eval_path = os.path.join(WORK_DIR, "eval_results.json")
with open(eval_path, "w") as f:
    json.dump(eval_results, f, indent=2)
print(f"\nEval results saved: {eval_path}")

# Final WRR summary
all_wrrs = [v["wrr_mean"] for v in eval_results.values()]
overall  = sum(all_wrrs) / len(all_wrrs)
print(f"\nOverall mean WRR: {overall:.4f}")
print(f"Promotion threshold: 0.70")
if overall >= 0.70:
    print("STATUS: ABOVE PROMOTION THRESHOLD")
else:
    print(f"STATUS: {0.70 - overall:.4f} below threshold — more training needed.")

## Section 11 — Anti-Hack Analysis

In [ ]:
# ============================================================
# CELL 14 — ANTI-HACK ANALYSIS
# Scans collected trajectories for 8 reward-hacking patterns.
# ============================================================

import sys
sys.path.insert(0, REPO_DIR)

from eval.anti_hack_checker import AntiHackChecker

print("Anti-Hack Pattern Analysis")
print("=" * 50)
print(f"Scanning {trajectory_buffer.get_stats()['buffer_size']} trajectories...")

top_trajectories = trajectory_buffer.get_top_n(n=50)

if not top_trajectories:
    print("Buffer is empty — no trajectories to scan.")
    print("(This is normal if GRPO_EPISODES was 0.)")
else:
    summary = AntiHackChecker.scan_trajectory_buffer(top_trajectories)

    print(f"\nSummary:")
    print(f"  Total scanned    : {summary['total_trajectories']}")
    print(f"  Clean (DPO-safe) : {summary['clean']}")
    print(f"  Flagged (review) : {summary['flagged_for_review']}")
    print(f"  Excluded (hack)  : {summary['excluded']}")

    if summary["pattern_frequency"]:
        print(f"\nDetected patterns:")
        for ptype, count in sorted(summary["pattern_frequency"].items(),
                                    key=lambda x: -x[1]):
            print(f"  {ptype:35s}: {count}")
        print(f"  Most common: {summary['most_common_pattern']}")
    else:
        print("  No hacking patterns detected.")

    dpo_safe_pct = summary['clean'] / summary['total_trajectories'] * 100
    print(f"\nDPO-safe rate: {dpo_safe_pct:.0f}%")
    if dpo_safe_pct < 50:
        print("WARNING: < 50% DPO-safe. Model may be reward hacking.")
        print("Consider more SFT epochs or reducing DPO learning rate.")

## Section 12 — Reward Curve Plots

In [ ]:
# ============================================================
# CELL 15 — GENERATE REWARD CURVE PLOTS
# ============================================================

import os, json
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

# Load episode log
log_path = os.path.join(WORK_DIR, "episode_log.json")
with open(log_path) as f:
    ep_log = json.load(f)

if len(ep_log) < 2:
    print("Not enough episodes for a meaningful plot. Run more GRPO_EPISODES.")
else:
    episodes   = [e["episode"]+1 for e in ep_log]
    wrrs       = [e["wrr"]       for e in ep_log]
    r1s        = [e["r1"]        for e in ep_log]
    r2s        = [e["r2"]        for e in ep_log]
    r3s        = [e["r3"]        for e in ep_log]
    qualities  = [e["quality"]   for e in ep_log]
    violations = [e["violations"]for e in ep_log]

    # ---- Figure 1: WRR + Component Breakdown ----
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle("QStorePrice AI — Training Metrics", fontsize=14, fontweight="bold")

    # WRR main curve
    ax = axes[0, 0]
    ax.plot(episodes, wrrs, "b-o", linewidth=2, markersize=5, label="WRR")
    ax.axhline(y=0.70, color="green", linestyle="--", alpha=0.8, label="Promotion threshold (0.70)")
    ax.axhline(y=0.08, color="red",   linestyle=":",  alpha=0.6, label="Zero-shot baseline (~0.08)")
    ax.fill_between(episodes, wrrs, 0.08, alpha=0.15, color="blue")
    ax.set_xlabel("Episode")
    ax.set_ylabel("WRR")
    ax.set_title("Weekly Waste Recovery Rate")
    ax.legend(fontsize=8)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)

    # Component breakdown
    ax = axes[0, 1]
    ax.plot(episodes, r1s, "r-^", markersize=4, label="r1 Pricing (w=0.40)")
    ax.plot(episodes, r2s, "g-s", markersize=4, label="r2 Farmer (w=0.30)")
    ax.plot(episodes, r3s, "b-D", markersize=4, label="r3 Trend (w=0.30)")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Reward")
    ax.set_title("Reward Component Breakdown")
    ax.legend(fontsize=8)
    ax.set_ylim(-0.1, 1.1)
    ax.grid(True, alpha=0.3)

    # Brief quality
    ax = axes[1, 0]
    ax.plot(episodes, qualities, "m-o", linewidth=2, markersize=5, label="Brief Quality")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Quality Score (0-1)")
    ax.set_title("Brief Quality Score (independent of WRR)")
    ax.set_ylim(0, 1.05)
    ax.axhline(y=0.7, color="orange", linestyle="--", alpha=0.6, label="Quality target")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Anti-hack violations
    ax = axes[1, 1]
    colors = ["red" if v > 3 else "orange" if v > 1 else "green" for v in violations]
    ax.bar(episodes, violations, color=colors, alpha=0.7)
    ax.set_xlabel("Episode")
    ax.set_ylabel("Violation Count")
    ax.set_title("Anti-Hack Violations per Episode")
    ax.axhline(y=3, color="red", linestyle="--", alpha=0.5, label="Warning threshold")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plot1_path = os.path.join(PLOTS_DIR, "training_metrics.png")
    plt.savefig(plot1_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Plot saved: {plot1_path}")

    # ---- Figure 2: Evaluation results bar chart ----
    if eval_results:
        fig2, ax2 = plt.subplots(figsize=(10, 5))
        sc_names  = list(eval_results.keys())
        sc_wrrs   = [eval_results[s]["wrr_mean"] for s in sc_names]
        sc_stds   = [eval_results[s]["wrr_std"]  for s in sc_names]
        bar_colors = ["green" if w >= 0.70 else "steelblue" for w in sc_wrrs]

        bars = ax2.bar(sc_names, sc_wrrs, color=bar_colors, alpha=0.8,
                       yerr=sc_stds, capsize=5, error_kw={"linewidth": 2})
        ax2.axhline(y=0.70, color="green",  linestyle="--", alpha=0.8, label="Promotion (0.70)")
        ax2.axhline(y=0.08, color="red",    linestyle=":",  alpha=0.6, label="Zero-shot baseline")

        for bar, wrr in zip(bars, sc_wrrs):
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f"{wrr:.3f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

        ax2.set_ylabel("WRR (Weekly Waste Recovery Rate)")
        ax2.set_title("Evaluation WRR per Scenario")
        ax2.set_ylim(0, 1.0)
        ax2.legend()
        ax2.grid(True, alpha=0.3, axis="y")

        plt.tight_layout()
        plot2_path = os.path.join(PLOTS_DIR, "eval_wrr_by_scenario.png")
        plt.savefig(plot2_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Plot saved: {plot2_path}")

## Section 13 — Backend Server

In [ ]:
# ============================================================
# CELL 16 — START FASTAPI SERVER (background)
# The server runs on localhost:8000 within the Kaggle kernel.
# External access isn't possible in Kaggle, but we can test
# all endpoints with internal HTTP requests.
# ============================================================

import sys, os, subprocess, time
sys.path.insert(0, REPO_DIR)

SERVER_PORT = 8000
SERVER_URL  = f"http://localhost:{SERVER_PORT}"

def start_server():
    """Attempt to start the FastAPI server.
    Returns the process handle or None on failure.
    """
    try:
        # Try full server first
        proc = subprocess.Popen(
            ["python", "-m", "server.app"],
            cwd=REPO_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            env={**os.environ, "PORT": str(SERVER_PORT), "WORKERS": "1"},
        )
        return proc, "full"
    except Exception:
        return None, None

def wait_for_server(url, retries=10, delay=2):
    """Poll the health endpoint until the server is up."""
    import urllib.request, urllib.error
    for i in range(retries):
        try:
            with urllib.request.urlopen(f"{url}/health", timeout=3) as r:
                if r.status == 200:
                    return True
        except Exception:
            pass
        time.sleep(delay)
        print(f"  Waiting for server... ({i+1}/{retries})")
    return False

print("Starting FastAPI backend server...")
server_proc, server_mode = start_server()

if server_proc:
    print(f"Server process started (PID={server_proc.pid}).")
    server_up = wait_for_server(SERVER_URL)
    if not server_up:
        # Print stderr for diagnosis
        server_proc.terminate()
        err = server_proc.stderr.read().decode()[:1000]
        print(f"Server failed to start. Stderr:\n{err}")
        print("\nFallback: running server API simulation inline.")
        server_proc = None
    else:
        print(f"Server is UP at {SERVER_URL}")
else:
    print("Could not start server process.")
    server_up = False

In [ ]:
# ============================================================
# CELL 17 — TEST ALL BACKEND ENDPOINTS
# Tests /health, /reset, /step, /state in sequence.
# If server is unavailable, uses a pure-Python simulation.
# ============================================================

import json, urllib.request, urllib.error

def http_get(url):
    try:
        with urllib.request.urlopen(url, timeout=10) as r:
            return json.loads(r.read())
    except Exception as e:
        return {"error": str(e)}

def http_post(url, data):
    try:
        req = urllib.request.Request(
            url, data=json.dumps(data).encode(), method="POST",
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(req, timeout=30) as r:
            return json.loads(r.read())
    except Exception as e:
        return {"error": str(e)}


if server_up:
    print("=" * 50)
    print(" BACKEND API TEST — Live Server")
    print("=" * 50)

    # 1. Health check
    resp = http_get(f"{SERVER_URL}/health")
    print(f"\nGET /health        → {resp}")

    # 2. Reset episode
    resp = http_post(f"{SERVER_URL}/reset", {"scenario": "STABLE_WEEK", "seed": 42})
    print(f"POST /reset        → keys: {list(resp.keys())}")
    obs_excerpt = str(resp.get("observation", ""))[:200]
    print(f"  observation[:200]: {obs_excerpt}")

    # 3. Step with a minimal brief
    step_payload = {
        "brief_text": (
            "## SITUATION SUMMARY\nBatch analysis complete.\n\n"
            "## SIGNAL ANALYSIS\nModerate expiry risk.\n\n"
            "## VIABILITY CHECK\nDiscount viable.\n\n"
            "## RECOMMENDATION\nHold prices on FRESH stock.\n\n"
            '## DIRECTIVE\n{"engine": "PRICING", "actions": []}\n\n'
            "## CONFIDENCE\n0.65\n"
        )
    }
    resp = http_post(f"{SERVER_URL}/step", step_payload)
    print(f"POST /step         → keys: {list(resp.keys())}")
    print(f"  reward           : {resp.get('reward', 'N/A')}")
    print(f"  done             : {resp.get('done', 'N/A')}")
    info_keys = list(resp.get('info', {}).keys())
    print(f"  info keys        : {info_keys}")

    # 4. State
    resp = http_get(f"{SERVER_URL}/state")
    print(f"GET /state         → {json.dumps(resp, indent=2)[:500]}")

    # 5. Docs endpoint
    resp = http_get(f"{SERVER_URL}/docs")
    print(f"GET /docs          → {'HTML page available' if 'error' not in resp else resp}")

    print(f"\nAll API endpoints tested. Server URL: {SERVER_URL}")
    print(f"Swagger UI: {SERVER_URL}/docs")

else:
    print("=" * 50)
    print(" BACKEND API — Python Simulation (no live server)")
    print("=" * 50)
    print("Server did not start (likely missing openenv-core).")
    print("Demonstrating equivalent logic via direct Python calls:")

    import sys
    sys.path.insert(0, REPO_DIR)
    from freshprice_env.freshprice_env import FreshPriceEnv
    from freshprice_env.enums import CurriculumScenario

    # Simulate GET /health
    print('\nSimulated GET /health → {"status": "ok"}')

    # Simulate POST /reset
    env = FreshPriceEnv(scenario=CurriculumScenario.STABLE_WEEK, seed=42)
    obs, info = env.reset()
    print(f"Simulated POST /reset → observation length={len(obs)}, engine={info['engine_type']}")

    # Simulate POST /step
    test_brief = (
        "## SITUATION SUMMARY\nInventory assessed.\n\n"
        "## SIGNAL ANALYSIS\nModerate demand.\n\n"
        "## VIABILITY CHECK\nConservative strategy.\n\n"
        "## RECOMMENDATION\nHold prices.\n\n"
        '## DIRECTIVE\n{"engine": "PRICING", "actions": []}\n\n'
        "## CONFIDENCE\n0.6\n"
    )
    obs2, reward, done, truncated, info2 = env.step(test_brief)
    print(f"Simulated POST /step → reward={reward:.4f}, done={done}")
    print(f"  parse_success={info2['parse_success']}, quality={info2['quality_score']:.3f}")

    # Simulate GET /state
    state = env.state()
    print(f"Simulated GET /state → {json.dumps(state, indent=2)[:400]}")

    print("\nTo run the live server locally:")
    print("  pip install openenv-core")
    print(f"  cd {REPO_DIR} && python -m server.app")

In [ ]:
# ============================================================
# CELL 18 — STOP SERVER (cleanup)
# ============================================================

if 'server_proc' in dir() and server_proc is not None:
    server_proc.terminate()
    server_proc.wait(timeout=5)
    print(f"Server process {server_proc.pid} terminated.")
else:
    print("No server process to terminate.")

## Section 14 — Push to Hugging Face Hub

In [ ]:
# ============================================================
# CELL 19 — PUSH TRAINED MODEL TO HUGGING FACE HUB
# Uses your HF token and HF_REPO_ID set in Cell 1.
# The model card is auto-generated with training metadata.
# ============================================================

import os, json
from huggingface_hub import HfApi, upload_folder, metadata_update

api = HfApi(token=HF_TOKEN)

# Create repo if it doesn't exist
print(f"Creating/verifying HF repo: {HF_REPO_ID}")
try:
    repo_url = api.create_repo(
        repo_id=HF_REPO_ID,
        repo_type="model",
        exist_ok=True,
        private=False,
    )
    print(f"Repo URL: {repo_url}")
except Exception as e:
    print(f"Repo creation: {e}")

# Determine what to push
push_dir = CURRENT_CHECKPOINT if os.path.exists(CURRENT_CHECKPOINT) else SFT_DIR
print(f"\nPushing from: {push_dir}")

# Write a model card
model_card = f"""---
tags:
  - qstoreprice
  - reinforcement-learning
  - perishable-goods
  - operating-brief
  - wrr
base_model: {MODEL_ID}
---

# QStorePrice AI — Trained Checkpoint

**Base model**: `{MODEL_ID}`  
**Training**: SFT warm-start ({SFT_EPOCHS} epochs) + GRPO rollouts ({GRPO_EPISODES} episodes)  
**Metric**: WRR (Weekly Waste Recovery Rate)  

## Evaluation Results

```json
{json.dumps(eval_results, indent=2)}
```

## Usage

```python
from freshprice_env import FreshPriceEnv
from freshprice_env.enums import CurriculumScenario

env = FreshPriceEnv(scenario=CurriculumScenario.STABLE_WEEK, seed=42)
obs, info = env.reset()
# Pass obs to your LLM, get back Operating Brief, then:
obs, reward, done, truncated, info = env.step(brief_text)
```

## Project
- GitHub: https://github.com/nandeshkanagaraju/QStorePrice  
- Trained on Kaggle with Unsloth 4-bit + LoRA
"""

card_path = os.path.join(push_dir, "README.md")
with open(card_path, "w") as f:
    f.write(model_card)
print("Model card written.")

# Push checkpoint folder to HF Hub
print(f"Uploading checkpoint to {HF_REPO_ID}...")
try:
    upload_folder(
        repo_id=HF_REPO_ID,
        folder_path=push_dir,
        repo_type="model",
        token=HF_TOKEN,
        ignore_patterns=["*.pyc", "__pycache__", "optimizer.pt"],
        commit_message=f"QStorePrice SFT+GRPO checkpoint — WRR={eval_results.get('STABLE_WEEK', {}).get('wrr_mean', 0):.4f}",
    )
    print(f"\nModel pushed successfully!")
    print(f"View at: https://huggingface.co/{HF_REPO_ID}")
except Exception as e:
    print(f"Push failed: {e}")
    print("Check your HF_TOKEN has write access and HF_REPO_ID is correct.")

# Also push eval plots
print("\nPushing evaluation plots...")
for plot_file in ["training_metrics.png", "eval_wrr_by_scenario.png"]:
    plot_path = os.path.join(PLOTS_DIR, plot_file)
    if os.path.exists(plot_path):
        try:
            api.upload_file(
                path_or_fileobj=plot_path,
                path_in_repo=f"plots/{plot_file}",
                repo_id=HF_REPO_ID,
                repo_type="model",
                token=HF_TOKEN,
            )
            print(f"  Uploaded: {plot_file}")
        except Exception as e:
            print(f"  {plot_file}: {e}")

## Section 15 — Final Summary

In [ ]:
# ============================================================
# CELL 20 — FINAL SUMMARY
# ============================================================

import os, json

print("=" * 65)
print(" QStorePrice AI — Run Complete")
print("=" * 65)

print(f"\n{'Model':<30}: {MODEL_ID}")
print(f"{'SFT epochs':<30}: {SFT_EPOCHS}")
print(f"{'GRPO episodes':<30}: {GRPO_EPISODES}")
print(f"{'Seed':<30}: {SEED}")
print(f"{'Final checkpoint':<30}: {CURRENT_CHECKPOINT}")
print(f"{'HF repo':<30}: https://huggingface.co/{HF_REPO_ID}")

print(f"\n{'─'*65}")
print(" Evaluation Results")
print(f"{'─'*65}")
if eval_results:
    print(f"  {'Scenario':<22} {'WRR':>8} {'±':>6} {'Quality':>8} {'Viol':>5} {'Const':>6}")
    print(f"  {'─'*56}")
    for sc, r in eval_results.items():
        print(
            f"  {sc:<22} {r['wrr_mean']:>8.4f} {r['wrr_std']:>6.4f} "
            f"{r['quality']:>8.4f} {r['violations_mean']:>5.1f} "
            f"{r['constitutional_pass_rate']:>6}"
        )
    all_wrrs = [v["wrr_mean"] for v in eval_results.values()]
    overall  = sum(all_wrrs) / len(all_wrrs)
    print(f"  {'─'*56}")
    print(f"  {'Overall mean WRR':<22} {overall:>8.4f}")
    gap = overall - 0.70
    if gap >= 0:
        print(f"  STATUS: ABOVE promotion threshold (+{gap:.4f})")
    else:
        print(f"  STATUS: {abs(gap):.4f} below promotion threshold (0.70)")

print(f"\n{'─'*65}")
print(" Output Files")
print(f"{'─'*65}")

outputs = [
    ("SFT checkpoint",         SFT_DIR),
    ("GRPO dir",               GRPO_DIR),
    ("DPO dir",                DPO_DIR),
    ("Episode log",            f"{WORK_DIR}/episode_log.json"),
    ("Eval results",           f"{WORK_DIR}/eval_results.json"),
    ("Training metrics plot",  f"{PLOTS_DIR}/training_metrics.png"),
    ("Eval WRR plot",          f"{PLOTS_DIR}/eval_wrr_by_scenario.png"),
]

for label, path in outputs:
    exists = os.path.exists(path)
    size_str = ""
    if exists and os.path.isdir(path):
        # Sum directory size
        total = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, fns in os.walk(path) for f in fns
        )
        size_str = f"({total/1e6:.0f} MB)"
    elif exists:
        size_str = f"({os.path.getsize(path)/1e3:.0f} KB)"

    status = f"OK {size_str}" if exists else "NOT CREATED (may have been skipped)"
    print(f"  {label:<30}: {status}")

print(f"\n{'─'*65}")
print(" What to do next")
print(f"{'─'*65}")
print("  1. Download checkpoint: Kaggle Output tab → checkpoints/")
print(f"  2. View on HF Hub:     https://huggingface.co/{HF_REPO_ID}")
print("  3. More training:      Increase GRPO_EPISODES (20+ recommended)")
print("  4. Better model:       Set MODEL_ID = Qwen/Qwen2.5-7B-Instruct")
print("  5. Full curriculum:    Run through all 5 CurriculumScenario levels")
print("  6. HF API inference:   Set USE_HF_INFERENCE_API=True (uses credits)")

print("\n" + "=" * 65)
print(" Run complete.")
print("=" * 65)